In [ ]:
# Cài các thư viện cần thiết (chạy lần đầu tiên)
!pip install pandas numpy lightgbm shap matplotlib seaborn tqdm plotly -q

print("Đã cài xong thư viện")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style("whitegrid")

print("Import thành công")

In [ ]:
from pathlib import Path
import os

data_path = r"C:\DATATHON ROUND1"

# Chuyển thành Path object (xử lý tốt hơn với tiếng Việt)
folder = Path(data_path)

print(f"Đường dẫn đang dùng: {folder}")

# Kiểm tra folder có tồn tại không
if folder.exists():
    print("Folder TỒN TẠI!")
    files_in_folder = list(folder.glob("*.csv"))
    print(f" Tìm thấy {len(files_in_folder)} file .csv trong folder:")
    for f in sorted(files_in_folder):
        print("   ✓", f.name)
else:
    print("KHÔNG tìm thấy folder")

if folder.exists():
    files = {
        'products.csv': {},
        'customers.csv': {'parse_dates': ['signup_date']},
        'promotions.csv': {'parse_dates': ['start_date', 'end_date']},
        'geography.csv': {},
        'orders.csv': {'parse_dates': ['order_date']},
        'order_items.csv': {},
        'payments.csv': {},
        'shipments.csv': {'parse_dates': ['ship_date', 'delivery_date']},
        'returns.csv': {'parse_dates': ['return_date']},
        'reviews.csv': {'parse_dates': ['review_date']},
        'sales.csv': {'parse_dates': ['Date']},
        'sample_submission.csv': {'parse_dates': ['Date']},
        'inventory.csv': {'parse_dates': ['snapshot_date']},
        'web_traffic.csv': {'parse_dates': ['date']},
    }

    data = {}
    for filename, kwargs in files.items():
        full_path = folder / filename
        try:
            df = pd.read_csv(full_path, **kwargs)
            data[filename.replace('.csv', '')] = df
            print(f" {filename:<20} | Shape: {df.shape}")
        except Exception as e:
            print(f" Lỗi load {filename}: {e}")

    print(f"\n HOÀN THÀNH! Đã load {len(data)} bảng dữ liệu.")
else:
    print("\n Chưa load được.")

In [ ]:
import pandas as pd
import numpy as np

print(" Đang tạo features...")

# Lấy các bảng cần thiết
sales = data['sales'].copy()
sample_sub = data['sample_submission'].copy()
orders = data['orders'].copy()
order_items = data['order_items'].copy()
promotions = data['promotions'].copy()
web = data['web_traffic'].copy()
inventory = data['inventory'].copy()

# sales train
sales = sales.rename(columns={'Date': 'date', 'Revenue': 'revenue', 'COGS': 'cogs'})
sales['date'] = pd.to_datetime(sales['date'])
sales = sales.set_index('date').sort_index()

# tạo features
def create_features(df):
    df = df.copy()
    
    # Time features
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month
    df['quarter'] = df.index.quarter
    df['year'] = df.index.year
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    
    # Lag & Rolling Revenue / COGS
    for lag in [1, 7, 14, 30, 60]:
        df[f'rev_lag_{lag}'] = df['revenue'].shift(lag)
        df[f'cogs_lag_{lag}'] = df['cogs'].shift(lag)
    
    for w in [7, 14, 30]:
        df[f'rev_roll_mean_{w}'] = df['revenue'].rolling(w).mean()
        df[f'rev_roll_std_{w}'] = df['revenue'].rolling(w).std()
    
    # Active promotions mỗi ngày
    promo_active = []
    for d in df.index:
        active = promotions[(promotions['start_date'] <= d) & (promotions['end_date'] >= d)].shape[0]
        promo_active.append(active)
    df['active_promos'] = promo_active
    
    # Web traffic
    web_daily = web.set_index('date')[['sessions', 'unique_visitors', 'page_views', 'bounce_rate']]
    df = df.join(web_daily, how='left')
    
    # 1. Gom nhóm theo ngày 
    inv_monthly = inventory.groupby('snapshot_date').agg({
        'stock_on_hand': 'mean',      # trung bình tồn kho
        'fill_rate': 'mean',          # trung bình fill rate
        'stockout_flag': 'max'        # 1 nếu có bất kỳ sản phẩm nào hết hàng
    }).sort_index()
    
    # 2. Forward fill theo index của df
    inv = inv_monthly.reindex(df.index, method='ffill')
    # ============================================================
    
    df = df.join(inv, how='left')
    
    # Order volume + GMV
    daily_orders = orders.groupby('order_date').agg(
        n_orders=('order_id', 'nunique'),
        n_customers=('customer_id', 'nunique')
    )
    
    item_value = order_items.groupby('order_id')['unit_price'].sum().reset_index()
    item_value = item_value.merge(orders[['order_id', 'order_date']], on='order_id')
    daily_gmv = item_value.groupby('order_date')['unit_price'].agg(['sum', 'mean']).rename(
        columns={'sum': 'daily_gmv', 'mean': 'avg_item_price'})
    
    daily_orders = daily_orders.join(daily_gmv)
    df = df.join(daily_orders, how='left')
    
    # Fill NaN
    df = df.fillna(0)
    return df

# Tạo train set
train_df = create_features(sales)

print("Feature Engineering hoàn thành!")
print(f"Train shape: {train_df.shape}")
print("Các cột:", train_df.columns.tolist())

In [ ]:
import lightgbm as lgb

print(" Đang train model...")

features = [col for col in train_df.columns if col not in ['revenue', 'cogs']]
target = 'revenue'

X = train_df[features]
y = train_df[target]

# Time series split
train_size = int(len(X) * 0.85)
X_train, X_val = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_val = y.iloc[:train_size], y.iloc[train_size:]

lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 64,
    'learning_rate': 0.05,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

model = lgb.train(params, lgb_train, 
                  num_boost_round=2000,
                  valid_sets=[lgb_val],
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])

print(" Model train xong!")

In [ ]:
print(" Đang dự báo 548 ngày...")

# Tính tỷ lệ COGS trung bình từ train
margin_ratio = (train_df['revenue'] - train_df['cogs']).mean() / train_df['revenue'].mean()
print(f"Margin ratio trung bình: {margin_ratio:.1%}")

def recursive_forecast(model, last_df, future_dates):
    preds = []
    current = last_df.copy()
    
    for date in future_dates:
        # Tạo row mới chỉ có index
        row = pd.DataFrame(index=[date])
        
        # Thêm 2 cột giả để hàm create_features cũ không bị KeyError
        row['revenue'] = 0.0
        row['cogs']    = 0.0
                
        # Gọi hàm create_features cũ 
        row = create_features(row)
        
        # Update lag & rolling từ current
        for lag in [1,7,14,30,60]:
            row[f'rev_lag_{lag}'] = current['revenue'].iloc[-lag] if len(current) >= lag else 0
            row[f'cogs_lag_{lag}'] = current['cogs'].iloc[-lag] if len(current) >= lag else 0
        
        for w in [7,14,30]:
            row[f'rev_roll_mean_{w}'] = current['revenue'].iloc[-w:].mean() if len(current) >= w else 0
            row[f'rev_roll_std_{w}'] = current['revenue'].iloc[-w:].std() if len(current) >= w else 0
        
        # Forward fill các feature khác
        for col in ['active_promos', 'sessions', 'unique_visitors', 'stock_on_hand', 
                    'fill_rate', 'n_orders', 'n_customers', 'daily_gmv', 'avg_item_price']:
            if col in current.columns:
                row[col] = current[col].iloc[-1]
        
        # Chỉ giữ các cột mà model cần
        row = row[features]
        
        pred_rev = model.predict(row)[0]
        pred_cogs = pred_rev * (1 - margin_ratio)
        
        preds.append((pred_rev, pred_cogs))
        
        # Cập nhật current
        new_row = pd.DataFrame({'revenue': [pred_rev], 'cogs': [pred_cogs]}, index=[date])
        current = pd.concat([current, new_row])
    
    return preds

# CHẠY DỰ BÁO
future_dates = sample_sub['Date']
last_buffer = train_df.iloc[-90:]          # buffer 90 ngày cuối
forecast_result = recursive_forecast(model, last_buffer, future_dates)

# Tạo submission
submission = sample_sub.copy()
submission['Revenue'] = [x[0] for x in forecast_result]
submission['COGS']    = [x[1] for x in forecast_result]

submission.to_csv('submission.csv', index=False)
print(" Tạo xong submission.csv !")
print(submission.head())

In [ ]:
from IPython.display import FileLink, display, HTML

# Tạo link tải trực tiếp
display(HTML(f"""
<p style="font-size:18px; color:green;">
    🎉 <b>HOÀN THÀNH PART 3!</b><br>
    File submission.csv đã được tạo và sẵn sàng nộp.
</p>
"""))

display(FileLink('submission.csv', result_html_prefix="<b>📥 Click vào đây để TẢI NGAY submission.csv về máy:</b> "))

print(f" File nằm tại: {os.getcwd()}\\submission.csv")